# Final project ver2.0 — 씬 설계

    **Final project ver2.0** — Windows + VS Code + Colab GPU.

    로컬 파일과 Colab 서버는 서로 다른 컴퓨터입니다. VS Code에서 `Select Kernel → Colab`을 선택하고 GPU를 확인하세요. 처음에는 `RUN_CONDITIONS=["D"]`로 기반을 확인합니다. **A·B·D는 제공하는 완성 코드**입니다. A/B에는 저장 VLA 모델 경로만 지정하면 됩니다. C는 학생의 루프를 연결합니다. 기존 `Final_project.ipynb`와 연구 조건의 의미가 다릅니다.

    씬·시작 명령·내부 프롬프트·루프 입출력의 자세한 순서는 **`docs/FINAL_PROJECT_V2_STUDENT_GUIDE.md`**를 먼저 읽으세요. 내부 프롬프트는 `final_project_v2/prompts/starter_strategy.txt`, C 루프는 `final_project_v2/student_strategy.py`입니다. B와 C는 기본적으로 같은 TXT를 읽습니다.

    기본 생성 씬은 시작 예시입니다. 기하 검사 결과는 실제 파지 불가능의 정답이 아닙니다. 실제 실행 영상으로 확인하세요. 안내는 `docs/FINAL_PROJECT_V2.md`에 있습니다.


In [ ]:
import os, sys, json, platform, subprocess, shutil, time, uuid
from pathlib import Path
IS_COLAB = platform.system() == 'Linux' and Path('/content').is_dir()
TEAM = 'team01'
SOURCE_ROOT = os.environ.get('FPV2_SOURCE', '')  # 업로드 후 실제 원격 vla_pipeline 경로
REPO_URL = 'https://github.com/semin126/VLA2026.git'
REPO_REF = 'main'                              # ver2.0이 push된 브랜치/commit
INSTALL = IS_COLAB and os.environ.get('FPV2_SKIP_INSTALL') != '1'
USE_DRIVE = IS_COLAB and os.environ.get('FPV2_NO_DRIVE') != '1'
PYTHON_OVERRIDE = os.environ.get('FPV2_PYTHON', '')
WORK = Path(os.environ.get('FPV2_WORK', '/content/Final_project_v2' if IS_COLAB else 'outputs/final_project_v2'))
SCENES_ROOT = os.environ.get('FPV2_SCENES', '')   # 같은 suite.json 재사용 위치
CUSTOM_SCENE = os.environ.get('FPV2_SCENE', '')    # 직접 저장한 씬 JSON. 비우면 개발 open
TASK_COMMAND = ''                              # 시작 명령. 비우면 scene.json의 command 사용
RUN_CONDITIONS = ['D']                         # A,B,C,D. E는 로컬 전용 셀
SECONDS = float(os.environ.get('FPV2_SECONDS', '12'))  # 연결 확인용. 연구 시 예비시험으로 결정
MAX_STEPS = int(os.environ.get('FPV2_MAX_STEPS', '2400'))
ACTION_STEPS = 240
OBSERVER_STEPS = 60
MAX_CALLS = 8
REPEAT_SEEDS = '42'
VLA_SCENE = ''                                 # A: 전체 과제 명령을 학습한 모델
VLA_SUBTASK = ''                               # B/C: 부분 운반 지시를 학습한 같은 모델
VLM_MODEL = 'HuggingFaceTB/SmolVLM2-500M-Video-Instruct'
STRATEGY_OVERRIDE = os.environ.get('FPV2_STRATEGY', '')
B_STRATEGY_OVERRIDE = os.environ.get('FPV2_B_STRATEGY', '')
FINAL_TEST = False                            # 전략·모델 확정 후에만 True


## 코드·실행 환경·저장 위치

아직 push하지 않은 코드는 `Upload to Colab` 후 `SOURCE_ROOT`로 지정합니다. 기존 작업 폴더를 자동 pull/reset하지 않습니다. Linux에서는 독립 Python 3.12 환경과 기존 잠금 파일을 사용합니다.

Drive는 각 명령 종료 시 해시를 확인하여 백업합니다. 세션이 도중에 사라지면 미복사 파일은 잃을 수 있습니다. 모델 로딩 시간은 과제 시간과 별도로 기록합니다.


In [ ]:
if SOURCE_ROOT:
    PROJECT = Path(SOURCE_ROOT).expanduser().resolve()
else:
    candidates = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates += [base, base/'vla_pipeline', base/'2026/VLA2026/vla_pipeline']
    PROJECT = next((p for p in candidates if (p/'scripts/final_project_v2.py').is_file()), None)
    if PROJECT is None:
        checkout = Path('/content' if IS_COLAB else '.') / ('fpv2_checkout_'+uuid.uuid4().hex[:8])
        subprocess.run(['git', 'clone', REPO_URL, str(checkout)], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', REPO_REF], check=True)
        PROJECT = checkout/'vla_pipeline'
PROJECT = PROJECT.resolve()
if not (PROJECT/'vlapipe/final_project_v2/runner.py').is_file():
    raise RuntimeError('ver2.0 코드가 없습니다. 최신 브랜치 또는 업로드 위치를 확인하세요.')
sys.path.insert(0, str(PROJECT))
from vlapipe.final_project_v2.notebook_runtime import NotebookRun, prepare_python
if not WORK.is_absolute():
    WORK = PROJECT/WORK
if USE_DRIVE:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
    BACKUP = Path('/content/drive/MyDrive/VLA2026/Final_project_v2')
else:
    BACKUP = None
run = NotebookRun(PROJECT, WORK, BACKUP, TEAM)
PY = prepare_python(PROJECT, WORK, install=INSTALL, python_override=PYTHON_OVERRIDE)
run.python = PY
B_STRATEGY = Path(B_STRATEGY_OVERRIDE) if B_STRATEGY_OVERRIDE else PROJECT/'final_project_v2/baseline_strategy.py'
C_STRATEGY = Path(STRATEGY_OVERRIDE) if STRATEGY_OVERRIDE else PROJECT/'final_project_v2/student_strategy.py'
DATA_STRATEGY = B_STRATEGY  # 같은 TXT 전략으로 초기 VLM 학습 후보 생성
print('PROJECT:', PROJECT, '\nPYTHON:', PY, '\nRUN:', run.root, '\nBACKUP:', run.backup)
run.run('scripts/check_env.py')


In [ ]:
if SCENES_ROOT:
    SCENES = Path(SCENES_ROOT).resolve()
else:
    SCENES = run.root/'scenes'
    run.run('scripts/final_project_v2.py', 'scenes', '--out', SCENES, '--tests', 10)
registry = json.loads((SCENES/'suite.json').read_text(encoding='utf-8'))
DEV_SCENES = [SCENES/r['path'] for r in registry['scenes'] if r['split']=='development']
print('Scenes:', SCENES, '\nDevelopment:', *DEV_SCENES, sep='\n')
from IPython.display import display, Image, HTML
import base64
def show_video(path):
    encoded = base64.b64encode(Path(path).read_bytes()).decode('ascii')
    display(HTML('<video controls width="700"><source src="data:video/mp4;base64,'+encoded+'" type="video/mp4"></video>'))
display(Image(filename=str(DEV_SCENES[0].with_suffix('')/'scene.png'), width=700))


## 학생의 씬 설계

open / blocked / mixed에서 시작합니다. 모든 정리 대상이 막혀 있더라도 먼저 옮길 수 있는 방해물이 필요합니다. 모든 물체가 모든 동작으로 이동 불가능한 씬을 해결할 수 있다고 가정하지 않습니다. 현재는 운반 스킬을 쓰므로 미학습 밀기를 요구하지 않습니다.

`layout`: `[x,y,yaw]`(m, rad), `research_targets`: 정리 대상, `category`: 설계 의도. 렌더·초기 관통·도달 범위·안정된 높이를 확인하고 D-state 또는 사람으로 실행하세요. 기하 검사와 실제 파지 결과가 다르면 둘 다 남깁니다.


In [ ]:
SOURCE_SCENE = DEV_SCENES[2]
editable = json.loads(SOURCE_SCENE.read_text(encoding='utf-8'))
editable.pop('initial_state', None)
editable.pop('state_sha256', None)
editable['split'] = 'development'
editable['id'] = 'student_mixed_'+uuid.uuid4().hex[:8]
# 같은 원본의 변형은 family를 유지하고 학습/시험에 나누지 않습니다.
# 예: editable['layout']['object_3'] = [0.0, 0.20, 1.5708]
EDITED = run.root/'edited_scene.json'
EDITED.write_text(json.dumps(editable, indent=2), encoding='utf-8')
FROZEN = run.root/'student_scene.json'
run.run('scripts/final_project_v2.py', 'freeze', '--scene', EDITED, '--out', FROZEN)
run.run('scripts/final_project_v2.py', 'preview', '--scene', FROZEN, '--out', run.root/'student_preview')
display(Image(filename=str(run.root/'student_preview/scene.png'), width=700))


In [7]:
# === 직접 만든 3개 씬 → validation suite ===
# accessibility_scene_pack_20261105 의 세 레이아웃(all_accessible / all_blocked / mixed)을
# final_project_v2 의 validation 씬으로 등록합니다. 좌표를 셀에 직접 담아 로컬 업로드 없이
# Colab 서버에서 freeze -> register 까지 수행합니다. 실행 후 출력되는 경로를
# 실행 노트북(Final_project_v2_0.ipynb) CELL 1 의 SCENES_ROOT 에 넣으세요.
VALID_LAYOUTS = {
    'valid_open': ('open', 2000, {
        'object_1': [-0.07, 0.19, 0.0], 'object_2': [0.07, 0.27, 0.0], 'object_3': [-0.07, 0.35, 0.0],
        'object_4': [0.07, 0.43, 0.0], 'object_5': [-0.07, 0.51, 0.0], 'object_6': [0.07, 0.19, 0.0],
        'object_7': [-0.07, 0.27, 0.0], 'object_8': [0.07, 0.35, 0.0], 'object_9': [-0.07, 0.43, 0.0],
        'object_10': [0.07, 0.51, 0.0]}),
    'valid_blocked': ('blocked', 2001, {
        'object_1': [-0.0275, 0.19, 0.0], 'object_2': [0.0275, 0.27, 0.0], 'object_3': [-0.0275, 0.35, 0.0],
        'object_4': [0.0275, 0.43, 0.0], 'object_5': [-0.0275, 0.51, 0.0], 'object_6': [0.0275, 0.19, 0.0],
        'object_7': [-0.0275, 0.27, 0.0], 'object_8': [0.0275, 0.35, 0.0], 'object_9': [-0.0275, 0.43, 0.0],
        'object_10': [0.0275, 0.51, 0.0]}),
    'valid_mixed': ('mixed', 2002, {
        'object_1': [-0.04, 0.19, 0.0], 'object_2': [0.04, 0.27, 0.7853981633974483], 'object_3': [-0.04, 0.35, 0.0],
        'object_4': [0.04, 0.43, 0.7853981633974483], 'object_5': [-0.04, 0.51, 0.0], 'object_6': [0.04, 0.19, 0.7853981633974483],
        'object_7': [-0.04, 0.27, 0.0], 'object_8': [0.04, 0.35, 0.7853981633974483], 'object_9': [-0.04, 0.43, 0.0],
        'object_10': [0.04, 0.51, 0.7853981633974483]}),
}
_OBJECTS = ['object_%d' % i for i in range(1, 11)]
_COMMAND = 'Tidy up the table: put every numbered object in its designated storage zone.'

valid_spec_dir = run.root/'valid_specs'; valid_spec_dir.mkdir(parents=True, exist_ok=True)
valid_frozen_dir = run.root/'valid_frozen'; valid_frozen_dir.mkdir(parents=True, exist_ok=True)
valid_frozen = []
for _name, (_category, _seed, _layout) in VALID_LAYOUTS.items():
    _spec = {'schema': 'final_project_scene_v2', 'id': '%s_%d' % (_name, _seed), 'seed': _seed,
             'family': _name, 'split': 'validation', 'category': _category, 'object_set': 'number',
             'count': 10, 'command': _COMMAND, 'research_targets': _OBJECTS, 'layout': _layout,
             'note': 'From accessibility_scene_pack_20261105; geometric diagnostic, not a grasp-impossibility proof.'}
    _sp = valid_spec_dir/(_name + '.json')
    _sp.write_text(json.dumps(_spec, indent=2), encoding='utf-8')
    _fr = valid_frozen_dir/(_name + '.json')
    if _fr.exists():
        _fr.unlink()                       # freeze 는 기존 출력 파일을 거부합니다
    run.run('scripts/final_project_v2.py', 'freeze', '--scene', _sp, '--out', _fr)
    valid_frozen.append(_fr)

VALID_SUITE = run.root/'my_valid_suite'
if VALID_SUITE.exists():
    shutil.rmtree(VALID_SUITE)             # register-scenes 는 기존 폴더를 거부합니다
_args = ['register-scenes', '--out', VALID_SUITE]
for _fr in valid_frozen:
    _args += ['--scene', _fr]
run.run('scripts/final_project_v2.py', *_args)
print('\n=== validation suite 준비 완료 ===')
print('실행 노트북 CELL 1 의 SCENES_ROOT 에 넣을 경로:')
print(str(VALID_SUITE))
run.sync()


[
  {
    "path": "validation/scene_0000.json",
    "family": "valid_open",
    "split": "validation",
    "state_sha256": "b60c6de505f8fab554faa14a77605b74df01e3d43a029b487d1351a95fdb4903",
    "preview": "validation/scene_0000/scene.png"
  },
  {
    "path": "validation/scene_0001.json",
    "family": "valid_blocked",
    "split": "validation",
    "state_sha256": "d06681bd2ee155a2552dd30433621de72577af9053453095d98e0867ad519a07",
    "preview": "validation/scene_0001/scene.png"
  },
  {
    "path": "validation/scene_0002.json",
    "family": "valid_mixed",
    "split": "validation",
    "state_sha256": "370d3810b314386de5a883a7ec9e62e0188cbe6aa57ae89df6fff2af47c6d0ca",
    "preview": "validation/scene_0002/scene.png"
  }
]

=== validation suite 준비 완료 ===
실행 노트북 CELL 1 의 SCENES_ROOT 에 넣을 경로:
/content/Final_project_v2/team01/20261105T123404Z_889906a6/my_valid_suite


## 선택 사항: 브라우저 편집 UI

Gradio 표에서 좌표를 바꾸고 4방향 렌더를 확인하여 새 개발 씬으로 저장합니다. JSON 편집·PNG 표시는 추가 패키지 없이 동작합니다. 로컬은 별도 UI 환경을 권장합니다.

```powershell
py -3.12 -m venv .venv-ui
.\.venv-ui\Scripts\python.exe -m pip install -r requirements-v2-ui.txt
.\.venv-ui\Scripts\python.exe scripts/final_project_v2.py edit --scene <scene.json> --out outputs/editor
```

Colab은 Gradio를 설치하고 `--share` 링크를 사용합니다.


In [ ]:
START_EDITOR = False
EDITOR_PYTHON = PY  # 로컬에서는 .venv-ui/Scripts/python.exe로 변경
if START_EDITOR:
    command = [EDITOR_PYTHON, str(PROJECT/'scripts/final_project_v2.py'), 'edit',
               '--scene', str(SOURCE_SCENE), '--out', str(run.root/'editor')]
    if IS_COLAB:
        command.append('--share')
    editor_log = run.root/'editor.log'
    with editor_log.open('w', encoding='utf-8') as f:
        editor_process = subprocess.Popen(command, cwd=PROJECT, stdout=f, stderr=subprocess.STDOUT,
            env=dict(os.environ, PYTHONUNBUFFERED='1', PYTHONIOENCODING='utf-8'),
            creationflags=getattr(subprocess, 'CREATE_NO_WINDOW', 0))
    time.sleep(3)
    print(editor_log.read_text(encoding='utf-8'))
    print('로컬: http://127.0.0.1:7860 / Colab: 로그의 URL. 종료: editor_process.terminate()')


In [ ]:
RUN_DEBUG = False
if RUN_DEBUG:
    run.run('scripts/final_project_v2.py', 'trial', '--condition', 'D', '--scene', FROZEN,
            '--out', run.root/'debug_d', '--seconds', SECONDS)
    show_video(run.root/'debug_d/process.mp4')
run.sync()
